In [ ]:


using ITensors
using Printf
using Random
using Colors, Plots


Random.seed!(1234)
N = 120
J1 = 1
J2= 0.43

Jfp=-10


# Create N spin-one degrees of freedom
#sites = siteinds("S=1/2", N)
# Alternatively can make spin-half sites instead
sites = siteinds("S=1/2", N)
sitesb = siteinds("S=1", N)

for i in 2:2:N
    sites[i]=sitesb[i]
end

# Input operator terms which define a Hamiltonian
os = OpSum()
for j in 1:(N - 4)
    
    if j % 2 == 0
        Jf = 0
    else
        Jf = Jfp
    end
    
    os += Jf, "Sz", j, "Sz", j + 1
    os += Jf*0.5, "S+", j, "S-", j + 1
    os += Jf*0.5, "S-", j, "S+", j + 1
    os += J1,"Sz", j, "Sz", j + 2
    os += J1*0.5, "S+", j, "S-", j + 2
    os += J1*0.5, "S-", j, "S+", j + 2
    os += J2,"Sz", j, "Sz", j + 4
    os += J2*0.5, "S+", j, "S-", j + 4
    os += J2*0.5, "S-", j, "S+", j + 4
    
    
end

os += Jfp,"Sz", N-3, "Sz", N-2
os += Jfp*0.5, "S+", N-3, "S-", N-2
os += Jfp*0.5, "S-", N-3, "S+", N-2
os += J1,"Sz", N-3, "Sz", N-1
os += J1*0.5, "S+", N-3, "S-", N-1
os += J1*0.5, "S-", N-3, "S+", N-1
os += J2,"Sz", N-3, "Sz", N
os += J2*0.5, "S+", N-3, "S-", N
os += J2*0.5, "S-", N-3, "S+", N

Jf=0

os += Jf,"Sz", N-2, "Sz", N-1
os += Jf*0.5, "S+", N-2, "S-", N-1
os += Jf*0.5, "S-", N-2, "S+", N-1
os += J1,"Sz", N-2, "Sz", N
os += J1*0.5, "S+", N-2, "S-", N
os += J1*0.5, "S-", N-2, "S+", N


    
    

os += Jfp,"Sz", N-1, "Sz", N
os += Jfp*0.5, "S+", N-1, "S-", N
os += Jfp*0.5, "S-", N-1, "S+", N






# Convert these terms to an MPO tensor network
H = MPO(os, sites)

# Create an initial random matrix product state
psi0 = randomMPS(sites; linkdims=20)

# Plan to do 5 DMRG sweeps:
nsweeps = 5
# Set maximum MPS bond dimensions for each sweep
maxdim = [150]
# Set maximum truncation error allowed when adapting bond dimensions
cutoff = [1E-12]

# Run the DMRG algorithm, returning energy and optimized MPS
energy, psi = dmrg(H, psi0; nsweeps, maxdim,cutoff=cutoff)
@printf("Final energy = %.12f\n", energy/N)






os = OpSum()
c = div(N, 2)
os += "Sz", c
kk = MPO(os, sites)

psi1= apply(kk,psi)

A=zeros(Float64, 2,N)
A=complex(A)






orthogonalize!(psi,1)

using ITensors.HDF5
f = h5open("spiin1by2andspin1,L="*string(N)*",BD="*string(maxdim[1])*"J2="*string(J2)*",Jf="*string(Jfp)*".h5","w")
write(f,"psi",psi)
close(f)


using ITensors.HDF5
f = h5open("spiin1by2andspin1,L="*string(N)*",J2="*string(J2)*",Jf="*string(Jfp)*".h5","w")
write(f,"mpo",H)
close(f)

H = nothing

psi= nothing

# Call garbage collection to free up memory
GC.gc()

